# Deepfake Detection Training Notebook

Tailored for Kaggle T4 GPU using mixed precision.

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix

# Ensure our modules are in path
import sys
sys.path.append('/kaggle/working/deepfake_detection')

from src.models.temporal_vit import AdvancedTemporalViT
from src.dataset import DeepfakeVideoDataset

In [ ]:
# Setup Dataset Paths (e.g., FaceForensics++ or Celeb-DF extracted faces)
DATASET_ROOT = '/kaggle/input/faceforensics/original_sequences/youtube/raw/frames'
FAKE_ROOT = '/kaggle/input/faceforensics/manipulated_sequences/Deepfakes/raw/frames'

# Collect paths and labels (Assuming directories contain frame images)
real_paths = glob.glob(os.path.join(DATASET_ROOT, '*'))[:200]  # Limit for example
fake_paths = glob.glob(os.path.join(FAKE_ROOT, '*'))[:200]

video_paths = real_paths + fake_paths
labels = [0]*len(real_paths) + [1]*len(fake_paths)

# Train / Val Split
from sklearn.model_selection import train_test_split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    video_paths, labels, test_size=0.2, random_state=42
)

print(f"Training samples: {len(train_paths)}")
print(f"Validation samples: {len(val_paths)}")

train_dataset = DeepfakeVideoDataset(train_paths, train_labels, sequence_length=8, augment=True)
val_dataset = DeepfakeVideoDataset(val_paths, val_labels, sequence_length=8, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

In [ ]:
# Model, Optimizer, Loss, and Scaler for Mixed Precision
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = AdvancedTemporalViT(use_mocks=False).to(device)
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()

os.makedirs('weights', exist_ok=True)

In [ ]:
EPOCHS = 5
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for x, res, label in train_loader:
        x, res, label = x.to(device), res.to(device), label.to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            probs = model(x, res).squeeze(-1)
        loss = criterion(probs.float(), label.float())
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item() * x.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss = 0.0
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for x, res, label in val_loader:
            x, res, label = x.to(device), res.to(device), label.to(device)
            
            with torch.amp.autocast('cuda'):
                probs = model(x, res).squeeze(-1)
            loss = criterion(probs.float(), label.float())
                
            val_loss += loss.item() * x.size(0)
            all_labels.extend(label.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    val_loss /= len(val_loader.dataset)
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'weights/best_model.pth')
        print("Saved new best model.")

In [ ]:
# Evaluation on Best Model
model.load_state_dict(torch.load('weights/best_model.pth'))
model.eval()

y_true, y_scores = [], []
with torch.no_grad():
    for x, res, label in val_loader:
        x, res = x.to(device), res.to(device)
        with torch.amp.autocast('cuda'):
            probs = model(x, res).squeeze(-1)
        y_true.extend(label.numpy())
        y_scores.extend(probs.cpu().numpy())
        
y_pred = [1 if p >= 0.5 else 0 for p in y_scores]

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()